In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder,LabelEncoder

In [2]:
df = pd.read_csv("../Data/dataset.csv")


In [ ]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import joblib



# 2. Clean column names (strip spaces)
df.columns = df.columns.str.strip()

# 3. Define symptom columns (Symptom_1 to Symptom_17)
symptom_cols = [f'Symptom_{i}' for i in range(1, 18)]

# 4. Function to clean text: strip spaces, lowercase, fix known typos
def clean_text(s):
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    # Fix common issues
    replacements = {
        'dischromic _patches': 'dischromic_patches',
        'spotting_ urination': 'spotting_urination',
        'toxic_look_(typhos)': 'toxic_look_typhos',
        'skin_rash': 'skin_rash',
        ' nodal_skin_eruptions': 'nodal_skin_eruptions',
        'continuous_sneezing': 'continuous_sneezing',
        'yellowish_skin': 'yellowish_skin',
        'dark_urine': 'dark_urine',
        'yellowing_of_eyes': 'yellowing_of_eyes',
        'loss_of_appetite': 'loss_of_appetite',
        'abdominal_pain': 'abdominal_pain',
        'passage_of_gases': 'passage_of_gases',
        'internal_itching': 'internal_itching',
        'muscle_wasting': 'muscle_wasting',
        'patches_in_throat': 'patches_in_throat',
        'high_fever': 'high_fever',
        'extra_marital_contacts': 'extra_marital_contacts',
        'blurred_and_distorted_vision': 'blurred_distorted_vision',
        'excessive_hunger': 'excessive_hunger',
        'increased_appetite': 'increased_appetite',
        'polyuria': 'polyuria',
        'sunken_eyes': 'sunken_eyes',
        'family_history': 'family_history',
        'mucoid_sputum': 'mucoid_sputum',
        'breathlessness': 'breathlessness',
        'fast_heart_rate': 'fast_heart_rate',
        'rusty_sputum': 'rusty_sputum',
        'swollen_blood_vessels': 'swollen_blood_vessels',
        'prominent_veins_on_calf': 'prominent_veins_on_calf',
        'cold_hands_and_feets': 'cold_hands_and_feet',
        'puffy_face_and_eyes': 'puffy_face_eyes',
        'brittle_nails': 'brittle_nails',
        'swollen_extremeties': 'swollen_extremities',
        'abnormal_menstruation': 'abnormal_menstruation',
        'drying_and_tingling_lips': 'drying_tingling_lips',
        'slurred_speech': 'slurred_speech',
        'palpitations': 'palpitations',
        'hip_joint_pain': 'hip_joint_pain',
        'painful_walking': 'painful_walking',
        'movement_stiffness': 'movement_stiffness',
        'spinning_movements': 'spinning_movements',
        'pus_filled_pimples': 'pus_filled_pimples',
        'blackheads': 'blackheads',
        'scurring': 'scurring',
        'bladder_discomfort': 'bladder_discomfort',
        'foul_smell_of urine': 'foul_smell_urine',
        'continuous_feel_of_urine': 'continuous_feel_urine',
        'skin_peeling': 'skin_peeling',
        'silver_like_dusting': 'silver_dusting',
        'small_dents_in_nails': 'small_dents_nails',
        'inflammatory_nails': 'inflammatory_nails',
        'red_sore_around_nose': 'red_sore_nose',
        'yellow_crust_ooze': 'yellow_crust_ooze',
        'receiving_blood_transfusion': 'blood_transfusion',
        'receiving_unsterile_injections': 'unsterile_injections',
        'acute_liver_failure': 'acute_liver_failure',
        'stomach_bleeding': 'stomach_bleeding',
        'swelling_of_stomach': 'swelling_stomach',
        'distention_of_abdomen': 'distended_abdomen',
        'history_of_alcohol_consumption': 'alcohol_history',
        'fluid_overload': 'fluid_overload',
        'blood_in_sputum': 'blood_in_sputum',
        'throat_irritation': 'throat_irritation',
        'redness_of_eyes': 'redness_eyes',
        'sinus_pressure': 'sinus_pressure',
        'runny_nose': 'runny_nose',
        'congestion': 'congestion',
        'loss_of_smell': 'loss_smell'
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    return s

# Apply cleaning to disease names and symptom columns
df['Disease'] = df['Disease'].apply(clean_text)
for col in symptom_cols:
    df[col] = df[col].apply(clean_text)

# 5. Create a list of symptoms per row (ignore empty strings)
df['symptom_list'] = df[symptom_cols].apply(
    lambda row: [s for s in row if s != ''], axis=1
)

# Remove rows that have no symptoms (should not happen, but safe)
df = df[df['symptom_list'].map(len) > 0]

# 6. MultiLabelBinarizer to create binary symptom features
mlb = MultiLabelBinarizer()
symptom_matrix = mlb.fit_transform(df['symptom_list'])
symptom_df = pd.DataFrame(symptom_matrix, columns=mlb.classes_)

# 7. Encode disease labels
le = LabelEncoder()
y = le.fit_transform(df['Disease'])

# 8. Combine features and label into one DataFrame (optional)
X = symptom_df.values
print(f"Feature matrix shape: {X.shape}")
print(f"Number of unique symptoms: {len(mlb.classes_)}")
print(f"Number of disease classes: {len(le.classes_)}")
print(f"Total samples: {len(y)}")

# 9. Remove duplicate rows (same symptom set + same disease)
full_df = pd.DataFrame(X, columns=mlb.classes_)
full_df['disease_code'] = y
full_df = full_df.drop_duplicates()
y_clean = full_df['disease_code'].values
X_clean = full_df.drop('disease_code', axis=1).values
print(f"After removing duplicates: {X_clean.shape[0]} samples")

# 10. Normalize features (important for scaling-sensitive models)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X_clean)

# 11. Train-test split (stratify to preserve class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

print("Preprocessing complete.")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# --- Save processed data and encoders into data/processed/ ---
np.save('../Data/processed/X_train.npy', X_train)
np.save('../Data/processed/X_test.npy', X_test)
np.save('../Data/processed/y_train.npy', y_train)
np.save('../Data/processed/y_test.npy', y_test)

joblib.dump(mlb, '../Models/symptom_binarizer.pkl')
joblib.dump(le, '../Models/disease_encoder.pkl')
joblib.dump(scaler, '../Models/feature_scaler.pkl')

print("Files saved to data/processed/")

Feature matrix shape: (4920, 131)
Number of unique symptoms: 131
Number of disease classes: 41
Total samples: 4920
After removing duplicates: 304 samples
Preprocessing complete.
Training set: 243 samples
Test set: 61 samples
Files saved to data/processed/
